# Thesis: Reclaimed Timber in Deep Generative Design
**Notebook ID:** 11_12_15_main_geometry_generation  
**Author:** Jasper Cluistra   
**Last Updated:** 2026-03-03
### Properties script
**Goal:** Main script that generates the geometry dataset that acts as input for the structural analysis in grasshopper, this script can take several variables as input at generates a CSV file made up of properties that can generate this digital geometry to a CAD geometry   
**Inputs:**   
*   Define base mesh
*   Allowed movement for different vertices
*   divisions over allowed movement

**Outputs:**

*   CSV file, with sample id, vertices, coordinates and edges

# 1.1 IMPORTING AND SETTING PARAMETERS

In [2]:
import pandas as pd
import numpy as np
import random

# --- CONFIGURATIE ---
GRID_CELLS_X = 1          # Aantal cellen in X
GRID_CELLS_Y = 1          # Aantal cellen in Y
EDGE_LENGTH = 3.0       # Afmeting van een cel
LAYER_HEIGHT = 1.5      # Afstand tussen top en bottom layer
DIVISIONS = 8             # Aantal stappen voor de discrete verplaatsing
NUM_SAMPLES = 10000       # Aantal samples
SCALE_UV = 0.25, 0.75        # Random positie in de cel

GRID = f"{GRID_CELLS_X}x{GRID_CELLS_Y}"

# 1.2 GEOMETRY GENERATION

## Structural Attribute of vertices
script om de support points te identificeren

In [3]:
def get_corner_indices(cells_x, cells_y):
    """
    Berekent de indices van de hoekpunten voor de Top Layer.
    Werkt voor elke grid grootte (n x m).
    """
    # Aantal punten is altijd aantal cellen + 1
    nodes_x = cells_x + 1
    nodes_y = cells_y + 1
    total_nodes = nodes_x * nodes_y

    indices = {
        "bottom_left": 0,
        "bottom_right": nodes_x - 1,
        "top_left": (nodes_y - 1) * nodes_x,
        "top_right": total_nodes - 1
    }

    return indices

In [4]:
# --- TESTEN ---
grid_2x2 = get_corner_indices(2, 2)
print(f"2x2 Grid Indices: {grid_2x2}")

grid_3x3 = get_corner_indices(3, 3)
print(f"3x3 Grid Indices: {grid_3x3}")

grid_4x4 = get_corner_indices(4, 4)
print(f"4x4 Grid Indices: {grid_4x4}")
# Verwacht: 0 (BL), 4 (BR), 20 (TL), 24 (TR) -> want 5 punten per rij

2x2 Grid Indices: {'bottom_left': 0, 'bottom_right': 2, 'top_left': 6, 'top_right': 8}
3x3 Grid Indices: {'bottom_left': 0, 'bottom_right': 3, 'top_left': 12, 'top_right': 15}
4x4 Grid Indices: {'bottom_left': 0, 'bottom_right': 4, 'top_left': 20, 'top_right': 24}


In [5]:
# In je loop:
corners = get_corner_indices(GRID_CELLS_X, GRID_CELLS_Y)
corner_values = corners.values() # [0, 4, 20, 24] bijv.
print(corner_values)

dict_values([0, 1, 2, 3])


## Calculating Vertices

In [6]:
def get_valid_shifts(divisions, edge_length):
    """Berekent de toegestane verschuivingen (verwijdert extremen)."""
    half_div = divisions // 2
    all_steps = list(range(-half_div, half_div + 1))
    valid_steps = all_steps[1:-1] # Verwijder eerste en laatste
    valid_shifts = [(step / divisions) * edge_length for step in valid_steps]
    return valid_shifts

def bilinear_interpolate(p00, p10, p01, p11, u, v):
    """
    Interpoleert een punt binnen een vierhoek.
    p00: Bottom-Left, p10: Bottom-Right, p01: Top-Left, p11: Top-Right (in standaard cartesiaans)
    Maar in matrix indexering (rij i, kol j):
    (i, j) is vaak Top-Left in images, maar Bottom-Left in Grasshopper/Cartesiaans als y omhoog gaat.
    Laten we aannemen: i=0 is y=0 (onder), i=max is y=max (boven).
    Dan is rij i "onder" rij i+1.
    p_bl = (i, j), p_br = (i, j+1)
    p_tl = (i+1, j), p_tr = (i+1, j+1)
    """
    # X interpolatie
    # Onderkant (row i)
    x_bot = p00['x'] * (1 - u) + p10['x'] * u
    # Bovenkant (row i+1)
    x_top = p01['x'] * (1 - u) + p11['x'] * u

    final_x = x_bot * (1 - v) + x_top * v

    # Y interpolatie
    y_bot = p00['y'] * (1 - u) + p10['y'] * u
    y_top = p01['y'] * (1 - u) + p11['y'] * u

    final_y = y_bot * (1 - v) + y_top * v

    return final_x, final_y

# --- Main Function ---
def generate_full_dataset(num_samples):
    valid_shifts = get_valid_shifts(DIVISIONS, EDGE_LENGTH)
    all_vertices = []

    num_nodes_x_top = GRID_CELLS_X + 1
    num_nodes_y_top = GRID_CELLS_Y + 1

    for sample_id in range(num_samples):
        # --- STAP 1: TOP LAYER ---
        top_layer_coords = {}
        vertex_idx = 0

        for i in range(num_nodes_y_top): # i is de rij index (y-richting)
            for j in range(num_nodes_x_top): # j is de kolom index (x-richting)
                # Basis positie
                base_x = j * EDGE_LENGTH
                base_y = i * EDGE_LENGTH
                base_z = 0.0

                # Structural Attribute Bepalen
                corners = get_corner_indices(GRID_CELLS_X, GRID_CELLS_Y)
                corner_values = corners.values()

                # Hier bepalen we het attribuut voor DIT punt
                if vertex_idx in corner_values:
                    attribute = "support"
                else:
                    attribute = "load"

                # Constraints
                is_x_edge = (j == 0) or (j == num_nodes_x_top - 1)
                is_y_edge = (i == 0) or (i == num_nodes_y_top - 1)
                is_corner = is_x_edge and is_y_edge

                shift_x = 0.0
                shift_y = 0.0

                if is_corner:
                    pass # Corner: Vast
                elif is_x_edge:
                    # Verticale rand (links/rechts): beweegt in Y
                    shift_y = random.choice(valid_shifts)
                elif is_y_edge:
                    # Horizontale rand (onder/boven): beweegt in X
                    shift_x = random.choice(valid_shifts)
                else:
                    # Inner: beweegt in X en Y
                    shift_x = random.choice(valid_shifts)
                    shift_y = random.choice(valid_shifts)

                final_x = base_x + shift_x
                final_y = base_y + shift_y
                final_z = base_z

                top_layer_coords[(i, j)] = {'x': final_x, 'y': final_y, 'z': final_z}

                all_vertices.append({
                    "sample_id": sample_id,
                    "vertex_index": f"v{vertex_idx}",
                    "layer": "top",
                    "attribute": attribute,
                    "x": round(final_x, 3),
                    "y": round(final_y, 3),
                    "z": round(final_z, 3)
                })
                vertex_idx += 1

        # --- STAP 2: LOWER LAYER ---
        # Vertex index loopt door vanaf 25
        # i loop 0..3, j loop 0..3

        for i in range(GRID_CELLS_Y):
            for j in range(GRID_CELLS_X):
                # We pakken de 4 hoekpunten van de top-cel
                # Rij i is 'onder', Rij i+1 is 'boven'
                p_bl = top_layer_coords[(i, j)]       # Bottom-Left
                p_br = top_layer_coords[(i, j+1)]     # Bottom-Right
                p_tl = top_layer_coords[(i+1, j)]     # Top-Left
                p_tr = top_layer_coords[(i+1, j+1)]   # Top-Right

                # Random positie in de cel (scaled 50%)
                u = random.uniform(*SCALE_UV)
                v = random.uniform(*SCALE_UV)

                # Interpoleren (let op volgorde parameters van mijn functie)
                # def bilinear_interpolate(p00, p10, p01, p11, u, v):
                # p00=BL, p10=BR, p01=TL, p11=TR
                lx, ly = bilinear_interpolate(p_bl, p_br, p_tl, p_tr, u, v)

                # Z positie
                base_z = -LAYER_HEIGHT
                z_shift = random.choice(valid_shifts)
                final_z = base_z + z_shift

                all_vertices.append({
                    "sample_id": sample_id,
                    "vertex_index": f"v{vertex_idx}",
                    "layer": "bottom",
                    "attribute": "hinges",
                    "x": round(lx, 3),
                    "y": round(ly, 3),
                    "z": round(final_z, 3)
                })
                vertex_idx += 1

    return pd.DataFrame(all_vertices)

In [7]:
def generate_edges(num_samples, cells_x, cells_y):
    edges_data = []

    # Bereken hulpparameters
    nodes_x_top = cells_x + 1
    nodes_y_top = cells_y + 1
    num_top_vertices = nodes_x_top * nodes_y_top

    # We itereren door elke sample om de edges per sample vast te leggen
    for sample_id in range(num_samples):

        edge_counter = 0  # Reset edge counter per sample (of wil je unieke ID's over de hele file? Meestal per sample resetten: e0..e127)

        # Hulpfunctie om edge toe te voegen
        def add_edge(u, v):
            nonlocal edge_counter
            edges_data.append({
                "sample_id": sample_id,
                "edge_id": f"e{edge_counter}",
                "V1": u,
                "V2": v,
            })
            edge_counter += 1

        # --- 1. TOP LAYER GRID ---
        # Vertices 0 tot num_top_vertices-1
        for r in range(nodes_y_top):      # loop rijen
            for c in range(nodes_x_top):  # loop kolommen
                current = r * nodes_x_top + c

                # Horizontaal (naar rechts)
                if c < cells_x: # zolang niet de laatste kolom
                    add_edge(current, current + 1)

                # Verticaal (naar beneden, of 'boven' in matrix index)
                if r < cells_y: # zolang niet de laatste rij
                    add_edge(current, current + nodes_x_top)

        # --- 2. BOTTOM LAYER GRID ---
        # Start index is na de laatste top vertex
        start_idx_bottom = num_top_vertices

        # Bottom grid heeft evenveel punten als er cellen zijn (cells_x * cells_y)
        # Maar de grid verbindingen zijn er eentje minder dan het aantal punten
        # Bottom punten zijn een grid van (cells_x) breed bij (cells_y) hoog.

        for r in range(cells_y):
            for c in range(cells_x):
                current = start_idx_bottom + r * cells_x + c

                # Horizontaal (naar rechts)
                if c < cells_x - 1:
                    add_edge(current, current + 1)

                # Verticaal (naar beneden)
                if r < cells_y - 1:
                    add_edge(current, current + cells_x)

        # --- 3. DIAGONALS (Pyramid connections) ---
        # Verbind elke Bottom vertex met de 4 Top vertices erboven
        for r in range(cells_y):
            for c in range(cells_x):
                bottom_node = start_idx_bottom + r * cells_x + c

                # De 4 corresponderende punten in de Top layer
                # Top grid is (cells_x + 1) breed
                top_tl = r * nodes_x_top + c               # Top-Left (of row i)
                top_tr = r * nodes_x_top + (c + 1)         # Top-Right
                top_bl = (r + 1) * nodes_x_top + c         # Bottom-Left (row i+1)
                top_br = (r + 1) * nodes_x_top + (c + 1)   # Bottom-Right

                add_edge(bottom_node, top_tl)
                add_edge(bottom_node, top_tr)
                add_edge(bottom_node, top_bl)
                add_edge(bottom_node, top_br)

    return pd.DataFrame(edges_data)

## Executing and saving

In [8]:
# --- EXECUTE ---
final_vertices = generate_full_dataset(NUM_SAMPLES)
final_edges = generate_edges(NUM_SAMPLES, GRID_CELLS_X, GRID_CELLS_Y)

# --- SAVE ---
final_vertices.to_csv(DATA_PATH + f"dataset_{GRID}_{NUM_SAMPLES}_vertices.csv", index=False)
final_edges.to_csv(DATA_PATH + f"dataset_{GRID}_{NUM_SAMPLES}_edges.csv", index=False)

print(f"Generatie gereed voor {NUM_SAMPLES} samples.")
print(f"Grid grootte: {GRID}")

print("Vertices Shape:", final_vertices.shape)
print("Edges Shape:", final_edges.shape)

print(f"\nTotaal aantal regels in vertices CSV: {len(final_vertices)}")
print("\nVoorbeeld output (eerste 5 regels van sample 0):")
print(final_vertices.head(5))
print("\nVoorbeeld output (eerste 5 regels van sample 1):")
print(final_vertices[final_vertices['sample_id'] == 1].head(5))

print(f"\nTotaal aantal regels in edges CSV: {len(final_edges)}")
print("\nVoorbeeld output (eerste 5 regels van sample 0):")
print(final_edges.head(5))
print("\nVoorbeeld output (eerste 5 regels van sample 1):")
print(final_edges[final_edges['sample_id'] == 1].head(5))

Generatie gereed voor 10000 samples.
Grid grootte: 1x1
Vertices Shape: (50000, 7)
Edges Shape: (80000, 4)

Totaal aantal regels in vertices CSV: 50000

Voorbeeld output (eerste 5 regels van sample 0):
   sample_id vertex_index   layer attribute      x      y     z
0          0           v0     top   support  0.000  0.000  0.00
1          0           v1     top   support  3.000  0.000  0.00
2          0           v2     top   support  0.000  3.000  0.00
3          0           v3     top   support  3.000  3.000  0.00
4          0           v4  bottom    hinges  1.475  1.882 -2.25

Voorbeeld output (eerste 5 regels van sample 1):
   sample_id vertex_index   layer attribute      x      y    z
5          1           v0     top   support  0.000  0.000  0.0
6          1           v1     top   support  3.000  0.000  0.0
7          1           v2     top   support  0.000  3.000  0.0
8          1           v3     top   support  3.000  3.000  0.0
9          1           v4  bottom    hinges  1.065

# 1.5 SEARCH SPACE

In [ ]:
import pandas as pd
import json

def define_search_space(cells_x, cells_y, divisions, edge_length):
    """
    Vertaalt de geometrische constraints naar een digitaal leesbare 'Search Space'
    voor een machine learning of optimalisatie algoritme.
    """
    valid_shifts = get_valid_shifts(divisions, edge_length)
    search_space = {}

    num_nodes_x_top = cells_x + 1
    num_nodes_y_top = cells_y + 1
    vertex_idx = 0

    # Hulpfunctie om te bepalen of een punt een hoek is
    corners = get_corner_indices(cells_x, cells_y).values()

    # --- 1. TOP LAYER VARIABELEN ---
    for i in range(num_nodes_y_top):
        for j in range(num_nodes_x_top):
            v_name = f"v{vertex_idx}"

            is_x_edge = (j == 0) or (j == num_nodes_x_top - 1)
            is_y_edge = (i == 0) or (i == num_nodes_y_top - 1)
            is_corner = vertex_idx in corners

            if is_corner:
                # AI mag hier niets mee doen
                pass
            elif is_x_edge:
                # Mag alleen schuiven over de Y-as
                search_space[f"{v_name}_shift_y"] = {"type": "discrete", "options": valid_shifts}
            elif is_y_edge:
                # Mag alleen schuiven over de X-as
                search_space[f"{v_name}_shift_x"] = {"type": "discrete", "options": valid_shifts}
            else:
                # Inner node: Mag in beide richtingen schuiven
                search_space[f"{v_name}_shift_x"] = {"type": "discrete", "options": valid_shifts}
                search_space[f"{v_name}_shift_y"] = {"type": "discrete", "options": valid_shifts}

            vertex_idx += 1

    # --- 2. BOTTOM LAYER VARIABELEN ---
    for r in range(cells_y):
        for c in range(cells_x):
            v_name = f"v{vertex_idx}"

            # u en v bepalen de positie ONDER de top-cel (continue waarden tussen 0.25 en 0.75)
            search_space[f"{v_name}_u"] = {"type": "continuous", "min": SCALE_UV[0], "max": SCALE_UV[1]}
            search_space[f"{v_name}_v"] = {"type": "continuous", "min": SCALE_UV[0], "max": SCALE_UV[1]}

            # Z-shift is weer een discrete stap
            search_space[f"{v_name}_shift_z"] = {"type": "discrete", "options": valid_shifts}

            vertex_idx += 1

    return search_space

# --- UITVOEREN EN BEKIJKEN ---

# Gebruik de variabelen uit je eerdere configuratie (bijv. 1x1 grid)
ai_search_space = define_search_space(GRID_CELLS_X, GRID_CELLS_Y, DIVISIONS, EDGE_LENGTH)

print(f"De Search Space bevat in totaal {len(ai_search_space)} onafhankelijke variabelen (knoppen waar de AI aan mag draaien).")
print("\nVoorbeeld van hoe de computer dit ziet:")

# Toon de eerste 5 variabelen mooi geformatteerd
for var_name, bounds in list(ai_search_space.items()):
    print(f"- {var_name}: {bounds}")

: 

In [10]:
# Sla de dictionary op als een JSON bestand
json_path = os.path.join(DATA_IMPORT_PATH, 'search_space.json')
with open(json_path, 'w') as f:
    json.dump(ai_search_space, f, indent=4) # indent=4 maakt het mooi leesbaar

print(f"✅ Search Space succesvol opgeslagen als '{json_path}'")

✅ Search Space succesvol opgeslagen als '/content/drive/MyDrive/Thesis_TU/data_import/search_space.json'
